<a href="https://colab.research.google.com/github/Estevamfla/b3-financial-intelligence/blob/main/02_analise_automatica_empresas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import requests
import zipfile
from io import BytesIO

# CONFIGURAÇÕES
ANO = 2025

url = (
    "https://dados.cvm.gov.br/dados/"
    f"CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{ANO}.zip"
)

print("Baixando dados da CVM...")

resposta = requests.get(url, timeout=120)
resposta.raise_for_status()

# LER DRE CONSOLIDADA
with zipfile.ZipFile(BytesIO(resposta.content)) as z:
    nome = f"dfp_cia_aberta_DRE_con_{ANO}.csv"

    with z.open(nome) as arquivo:
        dre_geral = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            dtype={
                "CNPJ_CIA": str,
                "CD_CONTA": str
            },
            low_memory=False
        )

# Selecionar exercício atual
dre_geral = dre_geral[
    (dre_geral["ORDEM_EXERC"] == "ÚLTIMO") &
    (dre_geral["DT_FIM_EXERC"] == f"{ANO}-12-31")
].copy()

# Selecionar última versão de cada empresa
ultima = dre_geral.groupby(
    ["CNPJ_CIA", "DT_REFER"]
)["VERSAO"].transform("max")

dre_geral = dre_geral[
    dre_geral["VERSAO"] == ultima
].copy()

# NORMALIZAR VALORES
fatores = {
    "MIL": 1000,
    "UNIDADE": 1,
    "REAL": 1
}

escala = (
    dre_geral["ESCALA_MOEDA"]
    .astype(str)
    .str.strip()
    .str.upper()
)

if not escala.isin(fatores).all():
    raise ValueError("Escala monetária desconhecida")

dre_geral["VALOR_REAIS"] = (
    pd.to_numeric(dre_geral["VL_CONTA"], errors="raise")
    * escala.map(fatores)
)

# CONTAS QUE VAMOS UTILIZAR
contas = {
    "3.01": "Receita",
    "3.03": "Lucro_Bruto",
    "3.11": "Lucro_Liquido"
}

base = dre_geral[
    dre_geral["CD_CONTA"].isin(contas)
].copy()

# Garantir uma conta de cada tipo por empresa
chaves = ["CNPJ_CIA", "DENOM_CIA", "CD_CONTA"]

duplicados = base.duplicated(chaves, keep=False)

if duplicados.any():
    print("Registros ambíguos:", duplicados.sum())
    print("Excluindo empresas com contas duplicadas.")

    cnpj_ambiguos = base.loc[
        duplicados, "CNPJ_CIA"
    ].unique()

    base = base[
        ~base["CNPJ_CIA"].isin(cnpj_ambiguos)
    ]

# TRANSFORMAR CONTAS EM COLUNAS
resumo_empresas = base.pivot(
    index=["CNPJ_CIA", "DENOM_CIA"],
    columns="CD_CONTA",
    values="VALOR_REAIS"
).reset_index()

resumo_empresas = resumo_empresas.rename(
    columns=contas
)

# Exigir contas necessárias
resumo_empresas = resumo_empresas.dropna(
    subset=["Receita", "Lucro_Bruto", "Lucro_Liquido"]
)

# Evitar divisão por zero
resumo_empresas = resumo_empresas[
    resumo_empresas["Receita"] > 0
].copy()

# INDICADORES AUTOMÁTICOS
resumo_empresas["Margem_Bruta_%"] = (
    resumo_empresas["Lucro_Bruto"]
    / resumo_empresas["Receita"] * 100
).round(2)

resumo_empresas["Margem_Liquida_%"] = (
    resumo_empresas["Lucro_Liquido"]
    / resumo_empresas["Receita"] * 100
).round(2)

resumo_empresas["Ano"] = ANO

# Organização
resumo_empresas = resumo_empresas.sort_values(
    "Receita", ascending=False
)

print("\nANÁLISE AUTOMÁTICA CONCLUÍDA")
print("Empresas analisadas:", len(resumo_empresas))

display(resumo_empresas.head(15))


Baixando dados da CVM...
Registros ambíguos: 15
Excluindo empresas com contas duplicadas.

ANÁLISE AUTOMÁTICA CONCLUÍDA
Empresas analisadas: 413


CD_CONTA,CNPJ_CIA,DENOM_CIA,Receita,Lucro_Bruto,Lucro_Liquido,Margem_Bruta_%,Margem_Liquida_%,Ano
244,33.000.167/0001-01,PETROLEO BRASILEIRO S.A. PETROBRAS,4.975490e+11,2.369980e+11,1.106050e+11,47.63,22.23,2025
51,02.916.265/0001-60,JBS S.A.,4.805830e+11,6.305880e+10,1.253591e+10,13.12,2.61,2025
299,49.115.815/0001-05,JBS N.V.,4.805830e+11,6.305880e+10,1.247410e+10,13.12,2.60,2025
0,00.000.000/0001-91,BCO BRASIL S.A.,3.194621e+11,1.010107e+11,1.678194e+10,31.62,5.25,2025
334,60.746.948/0001-12,BCO BRADESCO S.A.,2.701831e+11,7.633935e+10,2.392464e+10,28.25,8.85,2025
256,33.592.510/0001-54,VALE S.A.,2.135950e+11,7.470800e+10,1.181100e+10,34.98,5.53,2025
260,34.274.233/0001-02,VIBRA ENERGIA S/A,1.890750e+11,9.218000e+09,1.979000e+09,4.88,1.05,2025
63,03.853.896/0001-40,MARFRIG GLOBAL FOODS S.A.,1.639627e+11,2.018451e+10,7.626750e+08,12.31,0.47,2025
409,90.400.888/0001-42,BCO SANTANDER (BRASIL) S.A.,1.624946e+11,5.763432e+10,1.296512e+10,35.47,7.98,2025
251,33.256.439/0001-39,ULTRAPAR PARTICIPACOES S.A.,1.423695e+11,9.358841e+09,2.541908e+09,6.57,1.79,2025


In [2]:

nome_arquivo = "indicadores_empresas_2025.csv"

resumo_empresas.to_csv(
    nome_arquivo,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo criado:", nome_arquivo)
print("Empresas exportadas:", len(resumo_empresas))


Arquivo criado: indicadores_empresas_2025.csv
Empresas exportadas: 413


In [3]:

# Localizar Petrobras na base automática
validacao = resumo_empresas[
    resumo_empresas["CNPJ_CIA"] == "33.000.167/0001-01"
]

if len(validacao) != 1:
    raise ValueError(
        "Petrobras não encontrada unicamente na base."
    )

empresa = validacao.iloc[0]

# Valores validados anteriormente
esperado = {
    "Receita": 497_549_000_000,
    "Lucro_Bruto": 236_998_000_000,
    "Lucro_Liquido": 110_605_000_000
}

for indicador, valor in esperado.items():
    encontrado = empresa[indicador]

    assert abs(encontrado - valor) < 1, (
        f"Divergência em {indicador}: {encontrado}"
    )

print("VALIDAÇÃO PETROBRAS 2025")
print("-" * 40)
print("Receita: OK")
print("Lucro Bruto: OK")
print("Lucro Líquido: OK")
print("Margem Bruta:", empresa["Margem_Bruta_%"], "%")
print("Margem Líquida:", empresa["Margem_Liquida_%"], "%")
print("-" * 40)
print("VALIDAÇÃO CONCLUÍDA COM SUCESSO!")


VALIDAÇÃO PETROBRAS 2025
----------------------------------------
Receita: OK
Lucro Bruto: OK
Lucro Líquido: OK
Margem Bruta: 47.63 %
Margem Líquida: 22.23 %
----------------------------------------
VALIDAÇÃO CONCLUÍDA COM SUCESSO!


In [4]:

import sqlite3
import pandas as pd

# Criar banco de dados SQLite
conexao = sqlite3.connect(
    "b3_financial_intelligence.db"
)

cursor = conexao.cursor()

# Tabela de empresas
cursor.execute("""
CREATE TABLE IF NOT EXISTS empresas (
    cnpj TEXT PRIMARY KEY,
    nome TEXT NOT NULL
)
""")

# Tabela de indicadores
cursor.execute("""
CREATE TABLE IF NOT EXISTS indicadores_financeiros (
    cnpj TEXT NOT NULL,
    ano INTEGER NOT NULL,
    receita REAL,
    lucro_bruto REAL,
    lucro_liquido REAL,
    margem_bruta REAL,
    margem_liquida REAL,
    PRIMARY KEY (cnpj, ano),
    FOREIGN KEY (cnpj) REFERENCES empresas(cnpj)
)
""")

conexao.commit()

print("Banco de dados criado com sucesso!")


Banco de dados criado com sucesso!


In [5]:

# Preparar os dados
dados = resumo_empresas.copy()

# Garantir que não existem CNPJs repetidos
assert not dados["CNPJ_CIA"].duplicated().any()

# Cadastrar empresas
empresas_sql = [
    (str(linha.CNPJ_CIA), str(linha.DENOM_CIA))
    for linha in dados.itertuples(index=False)
]

cursor.executemany("""
INSERT INTO empresas (cnpj, nome)
VALUES (?, ?)
ON CONFLICT(cnpj) DO UPDATE SET
    nome = excluded.nome
""", empresas_sql)

# Cadastrar indicadores financeiros
indicadores_sql = [
    (
        str(linha.CNPJ_CIA),
        int(linha.Ano),
        float(linha.Receita),
        float(linha.Lucro_Bruto),
        float(linha.Lucro_Liquido),
        float(linha._asdict().get("Margem_Bruta_%", 0))
    )
    for linha in []
]

# Selecionar e renomear colunas para facilitar
colunas = [
    "CNPJ_CIA", "Ano",
    "Receita", "Lucro_Bruto", "Lucro_Liquido",
    "Margem_Bruta_%", "Margem_Liquida_%"
]

indicadores_sql = list(
    dados[colunas].itertuples(index=False, name=None)
)

cursor.executemany("""
INSERT INTO indicadores_financeiros (
    cnpj, ano, receita, lucro_bruto,
    lucro_liquido, margem_bruta, margem_liquida
)
VALUES (?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(cnpj, ano) DO UPDATE SET
    receita = excluded.receita,
    lucro_bruto = excluded.lucro_bruto,
    lucro_liquido = excluded.lucro_liquido,
    margem_bruta = excluded.margem_bruta,
    margem_liquida = excluded.margem_liquida
""", indicadores_sql)

conexao.commit()

print("Empresas cadastradas:", len(empresas_sql))
print("Indicadores inseridos:", len(indicadores_sql))


Empresas cadastradas: 413
Indicadores inseridos: 413


In [6]:

consulta = """
SELECT
    e.nome AS empresa,
    i.ano,
    i.receita,
    i.lucro_liquido,
    i.margem_bruta,
    i.margem_liquida
FROM indicadores_financeiros i
JOIN empresas e ON e.cnpj = i.cnpj
WHERE i.ano = 2025
ORDER BY i.receita DESC
LIMIT 10
"""

resultado = pd.read_sql_query(
    consulta,
    conexao
)

display(resultado)

# Conferir quantidade de registros
total = pd.read_sql_query("""
SELECT COUNT(*) AS total
FROM indicadores_financeiros
WHERE ano = 2025
""", conexao)

print(
    "Total de registros SQL:",
    total.loc[0, "total"]
)


,empresa,ano,receita,lucro_liquido,margem_bruta,margem_liquida
0,PETROLEO BRASILEIRO S.A. PETROBRAS,2025,4.975490e+11,1.106050e+11,47.63,22.23
1,JBS S.A.,2025,4.805830e+11,1.253591e+10,13.12,2.61
2,JBS N.V.,2025,4.805830e+11,1.247410e+10,13.12,2.60
3,BCO BRASIL S.A.,2025,3.194621e+11,1.678194e+10,31.62,5.25
4,BCO BRADESCO S.A.,2025,2.701831e+11,2.392464e+10,28.25,8.85
5,VALE S.A.,2025,2.135950e+11,1.181100e+10,34.98,5.53
6,VIBRA ENERGIA S/A,2025,1.890750e+11,1.979000e+09,4.88,1.05
7,MARFRIG GLOBAL FOODS S.A.,2025,1.639627e+11,7.626750e+08,12.31,0.47
8,BCO SANTANDER (BRASIL) S.A.,2025,1.624946e+11,1.296512e+10,35.47,7.98
9,ULTRAPAR PARTICIPACOES S.A.,2025,1.423695e+11,2.541908e+09,6.57,1.79


Total de registros SQL: 413


In [7]:

import pandas as pd

# Verificar total de empresas cadastradas
total_empresas = pd.read_sql_query("""
SELECT COUNT(*) AS total
FROM empresas
""", conexao)

# Verificar total de indicadores
total_indicadores = pd.read_sql_query("""
SELECT COUNT(*) AS total
FROM indicadores_financeiros
""", conexao)

# Verificar registros com valores ausentes
inconsistencias = pd.read_sql_query("""
SELECT
    COUNT(*) AS registros_incompletos
FROM indicadores_financeiros
WHERE receita IS NULL
   OR lucro_bruto IS NULL
   OR lucro_liquido IS NULL
   OR margem_bruta IS NULL
   OR margem_liquida IS NULL
""", conexao)

print("VALIDAÇÃO DO BANCO SQL")
print("-" * 40)
print("Empresas:", total_empresas.iloc[0, 0])
print("Indicadores:", total_indicadores.iloc[0, 0])
print("Registros incompletos:", inconsistencias.iloc[0, 0])

if (
    total_empresas.iloc[0, 0] == 413
    and total_indicadores.iloc[0, 0] == 413
    and inconsistencias.iloc[0, 0] == 0
):
    print("VALIDAÇÃO SQL CONCLUÍDA!")
else:
    print("ATENÇÃO: revisar os registros.")


VALIDAÇÃO DO BANCO SQL
----------------------------------------
Empresas: 413
Indicadores: 413
Registros incompletos: 0
VALIDAÇÃO SQL CONCLUÍDA!


In [8]:

from google.colab import files

# Garantir que todas as alterações estejam salvas
conexao.commit()

# Criar uma cópia consistente do banco
destino = sqlite3.connect(
    "b3_financial_intelligence_backup.db"
)

conexao.backup(destino)
destino.close()

# Baixar o banco para seu computador
files.download(
    "b3_financial_intelligence_backup.db"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>